In [33]:
import json
import numpy as np
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple
import pandas as pd

In [48]:
feature_df_name = 'fifth_feature_df_14_01_2026.csv'

training_df = pd.read_csv(feature_df_name)

training_df.head(5)

,SUBJECT_ID,HADM_ID,ICUSTAY_ID,GCS_max,GCS_mean,Lactate_min,Lactate_max,Lactate_mean,BUN_min,BUN_mean,...,HR_mean,HR_max,HR_std,RDW_max,RDW_mean,RDW_min,RDW_std,age_adj_comorbidity_score,MEANBP_min,MEANBP_mean
0,3,145834,211552,6.0,4.444444,1.3,8.8,3.800000,36.0,40.500000,...,98.925373,168.0,28.103334,15.7,15.400000,15.0,0.273861,16.0,40.000000,76.728071
1,6,107064,228232,6.0,6.000000,NaN,NaN,NaN,65.0,65.500000,...,84.791667,100.0,5.573334,17.0,16.450000,15.9,0.777817,12.0,72.666702,87.175435
2,9,150750,220597,5.0,2.600000,1.9,2.7,2.380000,17.0,18.500000,...,87.644737,111.0,8.162849,14.3,14.000000,13.8,0.264575,12.0,67.000000,98.289474
3,12,112213,232669,6.0,4.842105,1.6,15.1,8.716667,28.0,35.500000,...,83.208333,105.0,8.518311,14.9,14.625000,14.2,0.309570,7.0,73.000000,96.819444
4,17,194023,277042,6.0,5.615385,0.9,0.9,0.900000,10.0,10.666667,...,82.400000,114.0,11.049719,12.9,12.766667,12.6,0.152753,0.0,53.000000,69.714286


In [25]:
# Example JSON structure for ICU patient data
EXAMPLE_JSON = {
    "patient_id": "ICU_042",
    "admission_time": "2025-01-07T08:12:00",
    "current_time": "2025-01-09T10:00:00",  # Time when prediction is requested (must be ≥48h after admission)
    "age": 67,
    "age_adj_comorbidity_score": 5,
    "measurements": [
        # Vital signs (frequent measurements)
        {"timestamp": "2025-01-07T08:15:00", "heart_rate": -500000000, "systolic_bp": 145, "diastolic_bp": 88, "mean_bp": 107, "respiratory_rate": 22, "temperature": 38.4},
        {"timestamp": "2025-01-07T08:45:00", "heart_rate": 118, "systolic_bp": 138, "diastolic_bp": 82, "mean_bp": 101, "respiratory_rate": 24, "temperature": 38.1},
        {"timestamp": "2025-01-07T09:30:00", "heart_rate": 105, "systolic_bp": 152, "diastolic_bp": 90, "mean_bp": 111, "respiratory_rate": 20, "temperature": 37.9},
        {"timestamp": "2025-01-07T10:15:00", "heart_rate": 98, "systolic_bp": 142, "diastolic_bp": 85, "mean_bp": 104, "respiratory_rate": 18, "temperature": 37.6},
        
        # Lab values (less frequent)
        {"timestamp": "2025-01-07T09:00:00", "gcs": 14, "lactate": 2.3, "bun": 28, "bilirubin": 1.2, "albumin": 3.1, "alk_phos": 95, "pt": 13.5, "inr": 1.1, "phosphate": 3.2, "pao2": 85, "aptt": 32, "anion_gap": 14, "rdw": 14.2},
        {"timestamp": "2025-01-07T21:00:00", "gcs": 15, "lactate": 1.8, "bun": 26, "bilirubin": 1.0, "albumin": 3.3, "alk_phos": 88, "pt": 12.8, "inr": 1.0, "phosphate": 3.0, "pao2": 92, "aptt": 30, "anion_gap": 12, "rdw": 14.0},
        
        # Day 2 measurements
        {"timestamp": "2025-01-08T08:30:00", "heart_rate": 92, "systolic_bp": 135, "diastolic_bp": 80, "mean_bp": 98, "respiratory_rate": 16, "temperature": 37.2},
        {"timestamp": "2025-01-08T09:00:00", "gcs": 15, "lactate": 1.5, "bun": 24, "bilirubin": 0.9, "albumin": 3.5, "alk_phos": 82, "pt": 12.2, "inr": 0.9, "phosphate": 2.8, "pao2": 95, "aptt": 28, "anion_gap": 11, "rdw": 13.8},
        
        # More vitals throughout 48h period
        {"timestamp": "2025-01-08T14:00:00", "heart_rate": 88, "systolic_bp": 130, "diastolic_bp": 78, "mean_bp": 95, "respiratory_rate": 15, "temperature": 36.9},
        {"timestamp": "2025-01-08T20:00:00", "heart_rate": 85, "systolic_bp": 128, "diastolic_bp": 76, "mean_bp": 93, "respiratory_rate": 14, "temperature": 36.8},
        
        # Day 3 (within 48h)
        {"timestamp": "2025-01-09T08:00:00", "heart_rate": 82, "systolic_bp": 125, "diastolic_bp": 75, "mean_bp": 92, "respiratory_rate": 14, "temperature": 36.7},
    ]
}


In [49]:
# Example JSON structure for ICU patient data
EXAMPLE_JSON = {
    "patient_id": "ICU_042",
    "admission_time": "2025-01-07T08:12:00",
    "current_time": "2025-01-09T10:00:00",  # Time when prediction is requested (must be ≥48h after admission)
    "age": 67,
    "age_adj_comorbidity_score": 5,
    "measurements": [
        # Vital signs (frequent measurements)
        {"timestamp": "2025-01-07T08:15:00", "systolic_bp": 145, "diastolic_bp": 88, "mean_bp": 107, "respiratory_rate": 22, "temperature": 38.4},
        {"timestamp": "2025-01-07T08:45:00", "systolic_bp": 138, "diastolic_bp": 82, "mean_bp": 101, "respiratory_rate": 24, "temperature": 38.1},
        {"timestamp": "2025-01-07T09:30:00", "systolic_bp": 152, "diastolic_bp": 90, "mean_bp": 111, "respiratory_rate": 20, "temperature": 37.9},
        {"timestamp": "2025-01-07T10:15:00", "systolic_bp": 142, "diastolic_bp": 85, "mean_bp": 104, "respiratory_rate": 18, "temperature": 37.6},
        
        # Lab values (less frequent)
        {"timestamp": "2025-01-07T09:00:00", "gcs": 14, "bun": 28, "bilirubin": 1.2, "albumin": 3.1, "alk_phos": 95, "pt": 13.5, "inr": 1.1, "phosphate": 3.2, "pao2": 85, "aptt": 32, "anion_gap": 14, "rdw": 14.2},
        {"timestamp": "2025-01-07T21:00:00", "gcs": 15, "bun": 26, "bilirubin": 1.0, "albumin": 3.3, "alk_phos": 88, "pt": 12.8, "inr": 1.0, "phosphate": 3.0, "pao2": 92, "aptt": 30, "anion_gap": 12, "rdw": 14.0},
        
        # Day 2 measurements
        {"timestamp": "2025-01-08T08:30:00", "systolic_bp": 135, "diastolic_bp": 80, "mean_bp": 98, "respiratory_rate": 16, "temperature": 37.2},
        {"timestamp": "2025-01-08T09:00:00", "gcs": 15, "bun": 24, "bilirubin": 0.9, "albumin": 3.5, "alk_phos": 82, "pt": 12.2, "inr": 0.9, "phosphate": 2.8, "pao2": 95, "aptt": 28, "anion_gap": 11, "rdw": 13.8},
        
        # More vitals throughout 48h period
        {"timestamp": "2025-01-08T14:00:00", "systolic_bp": 130, "diastolic_bp": 78, "mean_bp": 95, "respiratory_rate": 15, "temperature": 36.9},
        {"timestamp": "2025-01-08T20:00:00", "systolic_bp": 128, "diastolic_bp": 76, "mean_bp": 93, "respiratory_rate": 14, "temperature": 36.8},
        
        # Day 3 (within 48h)
        {"timestamp": "2025-01-09T08:00:00", "systolic_bp": 125, "diastolic_bp": 75, "mean_bp": 92, "respiratory_rate": 14, "temperature": 36.7},
    ]
}


In [51]:
def validate_measurement_value(param: str, value: float) -> bool:
    """
    Validate if a measurement value is within acceptable clinical ranges.
    
    Args:
        param: Parameter name (e.g., 'heart_rate', 'temperature')
        value: Measured value
    
    Returns:
        bool: True if valid, False otherwise
    """
    validation_ranges = {
        'heart_rate': (0, 350),
        'respiratory_rate': (0, 300),
        'temperature': (26, 45),
        'anion_gap': (5, 50),
        'systolic_bp': (0, 375),
        'diastolic_bp': (0, 375),
        'mean_bp': (0, 300),
    }
    
    if param not in validation_ranges:
        return True  # No validation rule, accept the value
    
    min_val, max_val = validation_ranges[param]
    return min_val < value < max_val


def validate_48h_coverage(measurements: List[Dict], admission_time: datetime, current_time: datetime) -> Tuple[bool, str]:
    """
    Validate that enough time has passed since admission and measurements exist within the 48h window.
    
    Args:
        measurements: List of measurement dictionaries
        admission_time: ICU admission timestamp
        current_time: Current time when prediction is requested
    
    Returns:
        Tuple[bool, str]: (is_valid, message)
    """
    if not measurements:
        return False, "No measurements provided"
    
    # Check if at least 48 hours have passed since admission
    hours_since_admission = (current_time - admission_time).total_seconds() / 3600
    if hours_since_admission < 48:
        return False, f"Insufficient time since admission: {hours_since_admission:.1f} hours (need ≥48 hours)"
    
    # Define the 48-hour window
    window_start = admission_time
    window_end = admission_time + timedelta(hours=48)
    
    # Get measurements within the 48h window
    valid_measurements = [
        m for m in measurements 
        if window_start <= datetime.fromisoformat(m['timestamp']) <= window_end
    ]
    
    if not valid_measurements:
        return False, "No measurements within the 48-hour window after admission"
    
    # Check time coverage within the window
    timestamps = sorted([datetime.fromisoformat(m['timestamp']) for m in valid_measurements])
    first_measurement = timestamps[0]
    last_measurement = timestamps[-1]
    
    coverage_hours = (last_measurement - first_measurement).total_seconds() / 3600
    
    # Check measurement density for vital signs
    vital_measurements = [m for m in valid_measurements if 'heart_rate' in m or 'systolic_bp' in m]
    if len(vital_measurements) < 6:
        return False, f"Insufficient vital sign measurements: {len(vital_measurements)} (need ≥6)"
    
    # Check for lab values
    lab_measurements = [m for m in valid_measurements if 'gcs' in m or 'lactate' in m or 'bun' in m]
    if len(lab_measurements) < 2:
        return False, f"Insufficient lab measurements: {len(lab_measurements)} (need ≥2)"
    
    return True, f"Valid: {hours_since_admission:.1f}h since admission, {coverage_hours:.1f}h measurement coverage, {len(valid_measurements)} measurements in 48h window"


def extract_values(measurements: List[Dict], param: str, admission_time: datetime) -> List[float]:
    """Extract all valid values for a specific parameter within 48h window."""
    window_end = admission_time + timedelta(hours=48)
    values = []
    
    for m in measurements:
        timestamp = datetime.fromisoformat(m['timestamp'])
        if admission_time <= timestamp <= window_end and param in m and m[param] is not None:
            value = float(m[param])
            # Validate the measurement is within acceptable clinical range
            if validate_measurement_value(param, value):
                values.append(value)
    
    return values


def calculate_statistics(values: List[float]) -> Dict[str, Optional[float]]:
    """Calculate min, max, mean, std for a list of values."""
    if not values:
        return {'min': None, 'max': None, 'mean': None, 'std': None}
    
    arr = np.array(values)
    return {
        'min': float(np.min(arr)),
        'max': float(np.max(arr)),
        'mean': float(np.mean(arr)),
        'std': float(np.std(arr, ddof=1)) if len(arr) > 1 else 0.0
    }


def engineer_features(patient_data: Dict, training_df=None, use_median_imputation: bool = True) -> Dict:
    """
    Convert raw ICU measurements into model features.
    
    Args:
        patient_data: Dictionary containing patient_id, admission_time, current_time,
                     age, age_adj_comorbidity_score, and measurements list
        training_df: Optional pandas DataFrame with training data for median imputation.
                    Should contain columns matching the feature names.
        use_median_imputation: If True and training_df is provided, replace None values 
                              with median from training data
    
    Returns:
        Dictionary with all engineered features or error information
    """
    try:
        # Parse timestamps
        admission_time = datetime.fromisoformat(patient_data['admission_time'])
        
        # Check if current_time is provided
        if 'current_time' not in patient_data:
            return {
                'success': False,
                'error': 'Missing current_time field',
                'details': 'current_time must be provided to validate 48-hour requirement'
            }
        
        current_time = datetime.fromisoformat(patient_data['current_time'])
        measurements = patient_data['measurements']
        
        # Validate that current_time is after admission_time
        if current_time <= admission_time:
            return {
                'success': False,
                'error': 'Invalid timestamps',
                'details': f'current_time ({current_time}) must be after admission_time ({admission_time})'
            }
        
        # Validate 48-hour coverage
        is_valid, message = validate_48h_coverage(measurements, admission_time, current_time)
        if not is_valid:
            return {
                'success': False,
                'error': 'Insufficient data coverage',
                'details': message
            }
        
        # Map JSON parameter names to feature prefixes
        param_mapping = {
            'gcs': 'GCS',
            'lactate': 'Lactate',
            'bun': 'BUN',
            'bilirubin': 'Bilirubin',
            'albumin': 'Albumin',
            'alk_phos': 'AlkPhos',
            'pt': 'PT',
            'inr': 'INR',
            'phosphate': 'Phosphate',
            'pao2': 'PaO2',
            'aptt': 'aPTT',
            'anion_gap': 'AG',
            'systolic_bp': 'SYSBP',
            'diastolic_bp': 'DIASBP',
            'mean_bp': 'MEANBP',
            'respiratory_rate': 'RR',
            'temperature': 'TEMP',
            'heart_rate': 'HR',
            'rdw': 'RDW'
        }
        
        features = {}
        
        # Static features
        features['age'] = patient_data.get('age')
        
        # Validate age_adj_comorbidity_score
        comorbidity_score = patient_data.get('age_adj_comorbidity_score')
        if comorbidity_score is not None:
            # Must be integer between -19 and 89
            if not isinstance(comorbidity_score, (int, float)) or comorbidity_score < -19 or comorbidity_score > 89:
                return {
                    'success': False,
                    'error': 'Invalid age_adj_comorbidity_score',
                    'details': f'Score must be between -19 and 89, got: {comorbidity_score}'
                }
            features['age_adj_comorbidity_score'] = int(comorbidity_score)
        else:
            features['age_adj_comorbidity_score'] = None
        
        # Calculate statistics for each parameter
        for json_param, feature_prefix in param_mapping.items():
            values = extract_values(measurements, json_param, admission_time)
            stats = calculate_statistics(values)
            
            # Create feature names based on what statistics are needed for each parameter
            if feature_prefix == 'GCS':
                # GCS: max, mean only
                if stats['max'] is not None:
                    features[f'{feature_prefix}_max'] = stats['max']
                if stats['mean'] is not None:
                    features[f'{feature_prefix}_mean'] = stats['mean']
            
            elif feature_prefix in ['Lactate', 'BUN', 'Albumin', 'AlkPhos']:
                # These need min, max, mean
                if stats['min'] is not None:
                    features[f'{feature_prefix}_min'] = stats['min']
                if stats['max'] is not None:
                    features[f'{feature_prefix}_max'] = stats['max']
                if stats['mean'] is not None:
                    features[f'{feature_prefix}_mean'] = stats['mean']
            
            elif feature_prefix == 'Bilirubin':
                # Bilirubin: max, mean only
                if stats['max'] is not None:
                    features[f'{feature_prefix}_max'] = stats['max']
                if stats['mean'] is not None:
                    features[f'{feature_prefix}_mean'] = stats['mean']
            
            elif feature_prefix in ['PT', 'INR', 'aPTT']:
                # These need mean, min only
                if stats['mean'] is not None:
                    features[f'{feature_prefix}_mean'] = stats['mean']
                if stats['min'] is not None:
                    features[f'{feature_prefix}_min'] = stats['min']
            
            elif feature_prefix in ['Phosphate', 'PaO2']:
                # These need mean, max only
                if stats['mean'] is not None:
                    features[f'{feature_prefix}_mean'] = stats['mean']
                if stats['max'] is not None:
                    features[f'{feature_prefix}_max'] = stats['max']
            
            elif feature_prefix == 'RDW':
                # RDW needs max, mean, min, std
                if stats['max'] is not None:
                    features[f'{feature_prefix}_max'] = stats['max']
                if stats['mean'] is not None:
                    features[f'{feature_prefix}_mean'] = stats['mean']
                if stats['min'] is not None:
                    features[f'{feature_prefix}_min'] = stats['min']
                if stats['std'] is not None:
                    features[f'{feature_prefix}_std'] = stats['std']
            
            elif feature_prefix == 'AG':
                # Anion gap needs all statistics
                for stat_name, stat_value in stats.items():
                    if stat_value is not None:
                        features[f'{feature_prefix}_{stat_name}'] = stat_value
            
            elif feature_prefix == 'SYSBP':
                # SYSBP needs min, mean, std
                if stats['min'] is not None:
                    features[f'{feature_prefix}_min'] = stats['min']
                if stats['mean'] is not None:
                    features[f'{feature_prefix}_mean'] = stats['mean']
                if stats['std'] is not None:
                    features[f'{feature_prefix}_std'] = stats['std']
            
            elif feature_prefix == 'DIASBP':
                # DIASBP needs min, mean only
                if stats['min'] is not None:
                    features[f'{feature_prefix}_min'] = stats['min']
                if stats['mean'] is not None:
                    features[f'{feature_prefix}_mean'] = stats['mean']
            
            elif feature_prefix == 'HR':
                # Heart rate needs mean, max, std only
                if stats['mean'] is not None:
                    features[f'{feature_prefix}_mean'] = stats['mean']
                if stats['max'] is not None:
                    features[f'{feature_prefix}_max'] = stats['max']
                if stats['std'] is not None:
                    features[f'{feature_prefix}_std'] = stats['std']
            
            elif feature_prefix == 'RR':
                # Respiratory rate: min, max, mean
                if stats['min'] is not None:
                    features[f'{feature_prefix}_min'] = stats['min']
                if stats['max'] is not None:
                    features[f'{feature_prefix}_max'] = stats['max']
                if stats['mean'] is not None:
                    features[f'{feature_prefix}_mean'] = stats['mean']
            
            elif feature_prefix == 'TEMP':
                # Temperature: min, std
                if stats['min'] is not None:
                    features[f'{feature_prefix}_min'] = stats['min']
                if stats['std'] is not None:
                    features[f'{feature_prefix}_std'] = stats['std']
            
            elif feature_prefix == 'MEANBP':
                # Mean BP: min, mean
                if stats['min'] is not None:
                    features[f'{feature_prefix}_min'] = stats['min']
                if stats['mean'] is not None:
                    features[f'{feature_prefix}_mean'] = stats['mean']
        
        # Check for missing critical features
        expected_features = [
            'GCS_max', 'GCS_mean', 'Lactate_min', 'Lactate_max', 'Lactate_mean',
            'BUN_min', 'BUN_mean', 'BUN_max', 'Bilirubin_max', 'Bilirubin_mean',
            'Albumin_mean', 'Albumin_min', 'Albumin_max', 'AlkPhos_mean', 'AlkPhos_max',
            'AlkPhos_min', 'PT_mean', 'PT_min', 'INR_mean', 'INR_min',
            'Phosphate_mean', 'Phosphate_max', 'PaO2_mean', 'PaO2_max', 'aPTT_mean',
            'aPTT_min', 'AG_mean', 'AG_max', 'AG_min', 'AG_std',
            'SYSBP_min', 'SYSBP_mean', 'SYSBP_std', 'DIASBP_min', 'DIASBP_mean',
            'age', 'RR_mean', 'RR_max', 'RR_min', 'TEMP_std', 'TEMP_min', 'HR_mean',
            'HR_max', 'HR_std', 'RDW_max', 'RDW_mean', 'RDW_min', 'RDW_std',
            'age_adj_comorbidity_score', 'MEANBP_min', 'MEANBP_mean'
        ]
        
        missing_features = [f for f in expected_features if f not in features or features[f] is None]
        
        # Apply median imputation if requested and training_df is provided
        imputed_features = []
        if use_median_imputation and training_df is not None and missing_features:
            for feature_name in missing_features:
                if feature_name in training_df.columns:
                    median_value = training_df[feature_name].median()
                    if not np.isnan(median_value):
                        features[feature_name] = float(median_value)
                        imputed_features.append(feature_name)
        
        # Update missing features list after imputation
        still_missing = [f for f in expected_features if f not in features or features[f] is None]
        
        return {
            'success': True,
            'features': features,
            'validation_message': message,
            'missing_features': still_missing if still_missing else None,
            'imputed_features': imputed_features if imputed_features else None,
            'total_measurements': len(measurements),
            'patient_id': patient_data['patient_id']
        }
        
    except Exception as e:
        return {
            'success': False,
            'error': str(e),
            'details': 'Error during feature engineering'
        }

In [53]:
result = engineer_features(EXAMPLE_JSON, training_df=training_df)

result

{'success': True,
 'features': {'age': 67,
  'age_adj_comorbidity_score': 5,
  'GCS_max': 15.0,
  'GCS_mean': 14.666666666666666,
  'BUN_min': 24.0,
  'BUN_max': 28.0,
  'BUN_mean': 26.0,
  'Bilirubin_max': 1.2,
  'Bilirubin_mean': 1.0333333333333334,
  'Albumin_min': 3.1,
  'Albumin_max': 3.5,
  'Albumin_mean': 3.3000000000000003,
  'AlkPhos_min': 82.0,
  'AlkPhos_max': 95.0,
  'AlkPhos_mean': 88.33333333333333,
  'PT_mean': 12.833333333333334,
  'PT_min': 12.2,
  'INR_mean': 1.0,
  'INR_min': 0.9,
  'Phosphate_mean': 3.0,
  'Phosphate_max': 3.2,
  'PaO2_mean': 90.66666666666667,
  'PaO2_max': 95.0,
  'aPTT_mean': 30.0,
  'aPTT_min': 28.0,
  'AG_min': 11.0,
  'AG_max': 14.0,
  'AG_mean': 12.333333333333334,
  'AG_std': 1.5275252316519468,
  'SYSBP_min': 125.0,
  'SYSBP_mean': 136.875,
  'SYSBP_std': 9.203066259211033,
  'DIASBP_min': 75.0,
  'DIASBP_mean': 81.75,
  'MEANBP_min': 92.0,
  'MEANBP_mean': 100.125,
  'RR_min': 14.0,
  'RR_max': 24.0,
  'RR_mean': 17.875,
  'TEMP_min': 36.7